# Training a unigram vocabulary

A unigram tokenizer gives every token a probability and splits a word the most probable
way. Its vocabulary is learned from the top down: training starts from a large set of
candidate tokens and alternates two steps, re-estimating the probabilities from the corpus
and then dropping the tokens the corpus needs least. You will write the probability of a
word under the model, the re-estimation step and the pruning step, and then train a
vocabulary on a small corpus of English words.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO`
and the `raise NotImplementedError` with your code, then run the check cell under it. The
check prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It holds the corpus, the seed vocabulary training starts from, and
the Viterbi split from the lesson.

In [ ]:
import math
from collections import Counter

# Each word with the number of times it appears in the corpus.
corpus = {
    "replay": 6, "replays": 3, "played": 9, "player": 7, "players": 5, "playing": 8,
    "rewrite": 4, "rewrites": 2, "writer": 6, "writers": 3, "writing": 7,
    "reread": 3, "reader": 6, "readers": 4, "reading": 9,
    "unread": 2, "unplayed": 2, "unwritten": 1,
}


def seed_vocabulary(corpus, max_len=6, min_count=2):
    """Every substring of up to max_len letters seen at least min_count times, and every
    letter, with probability proportional to how often it occurs in the corpus."""
    counts = Counter()
    for word, n in corpus.items():
        for i in range(len(word)):
            for j in range(i + 1, min(len(word), i + max_len) + 1):
                counts[word[i:j]] += n
    kept = {t: c for t, c in counts.items() if len(t) == 1 or c >= min_count}
    total = sum(kept.values())
    return {t: c / total for t, c in kept.items()}


def viterbi(word, p):
    """The most probable split of word under the token probabilities p, or None."""
    best = [0.0] + [-math.inf] * len(word)
    back = [0] * (len(word) + 1)
    for end in range(1, len(word) + 1):
        for start in range(end):
            piece = word[start:end]
            if piece in p and best[start] + math.log(p[piece]) > best[end]:
                best[end], back[end] = best[start] + math.log(p[piece]), start
    if best[-1] == -math.inf:
        return None
    pieces, end = [], len(word)
    while end > 0:
        pieces.append(word[back[end]:end])
        end = back[end]
    return pieces[::-1]


seed = seed_vocabulary(corpus)
print(len(corpus), "distinct words,", sum(corpus.values()), "in all;", len(seed), "tokens in the seed vocabulary")

## Part 1 (coding task): The probability of a word

A segmentation of a word into tokens has the product of its tokens' probabilities as its
probability, and the model's probability of the word is the sum of that over every
segmentation into tokens of the vocabulary. Write `word_prob(word, p)`, where `p` maps each
token to its probability, and return that sum, or `0.0` when the word has no segmentation.

In [ ]:
def word_prob(word, p):
    """Total probability of word over all its segmentations into tokens of p."""
    # TODO
    raise NotImplementedError

In [ ]:
p_small = {"a": 0.5, "b": 0.3, "ab": 0.2}
assert math.isclose(word_prob("ab", p_small), 0.35), f"word_prob('ab') = {word_prob('ab', p_small)}, expected 0.5 * 0.3 + 0.2 = 0.35"
assert math.isclose(word_prob("aab", p_small), 0.175), "aab: a + a + b and a + ab give 0.075 + 0.1 = 0.175"
assert math.isclose(word_prob("abab", p_small), 0.35 ** 2), "abab has four segmentations, which add up to 0.35 squared"
assert word_prob("abc", p_small) == 0, "abc cannot be split into tokens of p, so its probability is 0"
assert math.isclose(word_prob("ab" * 12, p_small), 0.35 ** 12), "wrong probability for a word of 24 letters with 4,096 segmentations"
assert math.isclose(word_prob("player", seed), 0.006832222664840395, rel_tol=1e-9), "wrong probability of player under the seed vocabulary"
print("Part 1 passed")

## Part 2 (coding task): Re-estimating the probabilities

Given the word, a segmentation $s$ has probability $P(s) / P(\text{word})$, where $P(s)$ is
the product of its token probabilities and $P(\text{word})$ is `word_prob`. Write two
functions.

- `expected_counts(corpus, p)` returns a dict mapping each token to the number of times
  the corpus uses it on average: each word's segmentations are weighted by their
  probability given the word, and each word counts as many times as it appears in
  `corpus`. A token that no segmentation uses may be left out.
- `em_step(corpus, p)` returns new probabilities that sum to 1, each token's in proportion
  to its expected count.

Repeating `em_step` is the expectation-maximization (EM) algorithm, and no step lowers the
probability of the corpus.

In [ ]:
def expected_counts(corpus, p):
    # TODO
    raise NotImplementedError


def em_step(corpus, p):
    # TODO
    raise NotImplementedError

In [ ]:
def near(counts, expected, what):
    for t, v in expected.items():
        got = counts.get(t, 0.0)
        assert math.isclose(got, v, rel_tol=1e-9, abs_tol=1e-12), f"{what}: token {t!r} has {got:.6f}, expected {v:.6f}"

c = expected_counts({"ab": 7}, p_small)
assert not math.isclose(c.get("ab", 0.0), 7.0), "ab is counted 7 times, as if every copy of the word used its best split: weight each segmentation by its probability given the word"
near(c, {"a": 3.0, "b": 3.0, "ab": 4.0}, "corpus {'ab': 7}")
near(expected_counts({"abab": 1, "b": 2}, p_small), {"a": 6 / 7, "ab": 8 / 7, "b": 20 / 7}, "corpus {'abab': 1, 'b': 2}")
near(em_step({"ab": 7}, p_small), {"a": 0.3, "b": 0.3, "ab": 0.4}, "em_step on {'ab': 7}")
q = em_step(corpus, seed)
assert math.isclose(sum(q.values()), 1.0), "the probabilities from em_step should sum to 1"
assert math.isclose(q["ing"], 0.05616446560621389, rel_tol=1e-6), f"after one EM step on the corpus, ing has probability {q['ing']:.6f}, expected 0.056164"
before = sum(n * math.log(word_prob(w, seed)) for w, n in corpus.items())
after = sum(n * math.log(word_prob(w, q)) for w, n in corpus.items())
print(f"log-probability of the corpus: {before:.1f} with the seed, {after:.1f} after one EM step")
print("Part 2 passed")

## Part 3 (coding task): Pruning

Write two functions.

- `removal_loss(corpus, p, token)` returns how much the log-probability of the corpus,
  $\sum_w n_w \ln P(w)$ with $n_w$ the count of word $w$, falls when `token` is removed
  from the vocabulary and every other token keeps its probability. Return `math.inf` when
  some word of the corpus can no longer be split.
- `prune(corpus, p, n_remove)` removes the `n_remove` tokens of two or more letters with the
  smallest removal loss and returns the remaining probabilities, scaled to sum to 1. Single
  letters always stay, so that any word spelled with the corpus's letters can still be
  encoded.

The check then trains a vocabulary on the corpus from the setup. Starting from the seed, it
runs an EM step and prunes a quarter of the multi-letter tokens, round after round, until 20
are left, and splits a few words the corpus does not contain.

In [ ]:
def removal_loss(corpus, p, token):
    # TODO
    raise NotImplementedError


def prune(corpus, p, n_remove):
    # TODO
    raise NotImplementedError

In [ ]:
assert math.isclose(removal_loss({"ab": 7}, p_small, "ab"), 7 * math.log(0.35 / 0.15)), "removing ab from p_small should cost 7 ln(0.35 / 0.15): the 7 copies of ab keep only the split a + b"
try:
    lost = removal_loss({"ab": 3}, {"a": 0.6, "ab": 0.4}, "ab")
except (ValueError, ZeroDivisionError):
    raise AssertionError("with only the tokens a and ab, removing ab leaves no split of the word ab: return math.inf")
assert lost == math.inf, "with only the tokens a and ab, removing ab leaves no split of the word ab: return math.inf"
assert removal_loss({"ab": 7}, p_small, "zz") == 0, "a token no word contains costs nothing to remove"

p_test = {"a": 0.2, "b": 0.2, "c": 0.15, "z": 0.05, "ab": 0.2, "bc": 0.15, "abc": 0.05}
c_test = {"abc": 10, "ab": 1}
assert math.isclose(removal_loss(c_test, p_test, "bc"), 10 * math.log(0.116 / 0.086)), "wrong removal loss for bc: the word abc falls from 0.116 to 0.086, and it appears 10 times"
kept = prune(c_test, p_test, 1)
assert "z" in kept, "prune removed the letter z: single letters always stay"
assert "abc" in kept, "prune removed abc: it is the least probable multi-letter token, but it is the best split of abc, the corpus's commonest word, so removing it costs the most. Remove the tokens with the smallest removal loss"
assert set(kept) == {"a", "b", "c", "z", "ab", "abc"}, f"prune kept {sorted(kept)}: removing bc costs the least"
assert math.isclose(sum(kept.values()), 1.0), "the probabilities left after pruning should sum to 1"
assert math.isclose(kept["ab"], 0.2 / 0.85), "scale the remaining probabilities by the same factor"
assert set(prune(c_test, p_test, 2)) == {"a", "b", "c", "z", "abc"}, "with n_remove = 2, bc and ab cost the least"

letters = sorted(set("".join(corpus)))
p = seed
while True:
    p = em_step(corpus, p)
    p = {t: q for t, q in p.items() if q > 0}         # EM can drive a token to exactly 0
    p = {**{ch: 1e-6 for ch in letters}, **p}        # every letter stays available
    total = sum(p.values())
    p = {t: q / total for t, q in p.items()}
    multi = [t for t in p if len(t) > 1]
    if len(multi) <= 20:
        break
    p = prune(corpus, p, max(1, min(len(multi) // 4, len(multi) - 20)))

print("multi-letter tokens, most probable first:")
print(" ", sorted(multi, key=lambda t: -p[t]))
for word in ["replaying", "rewriter", "unreader", "rereading", "unwriting", "replayer"]:
    print(f"  {word:<10} {' + '.join(viterbi(word, p))}")
assert len(multi) == 20 and {"ing", "re", "un", "play", "read", "writ"} <= set(multi), "the trained vocabulary is missing tokens a correct run keeps: ing, re, un, play, read, writ"
assert viterbi("unwriting", p) == ["un", "writ", "ing"], "a correct run splits unwriting as un + writ + ing"
print("Part 3 passed")

## Solution

### Part 1: The probability of a word

In [ ]:
def word_prob(word, p):
    """Total probability of word over all its segmentations into tokens of p."""
    total = [1.0] + [0.0] * len(word)        # total[i]: probability of the first i letters
    for end in range(1, len(word) + 1):
        for start in range(end):
            piece = word[start:end]
            if piece in p:
                total[end] += total[start] * p[piece]
    return total[-1]

**Why it works.** The table `total[i]` holds the probability of the first `i` letters, summed over all their
segmentations. Every segmentation of the first `end` letters is a segmentation of some
shorter prefix followed by one last token, and the sum splits the same way, so
`total[end]` adds `total[start] * p[piece]` over the tokens that end there. It is Viterbi
with a sum in place of the maximum, and it costs $O(L^2)$ for a word of $L$ letters,
however many segmentations there are: the word of 24 letters in the check has 4,096.
Viterbi alone would give 0.2 for `"ab"`, the probability of its best split.

### Part 2: Re-estimating the probabilities

In [ ]:
def expected_counts(corpus, p):
    counts = Counter()
    for word, n in corpus.items():
        L = len(word)
        fwd = [1.0] + [0.0] * L              # fwd[i]: probability of word[:i]
        for end in range(1, L + 1):
            for start in range(end):
                if word[start:end] in p:
                    fwd[end] += fwd[start] * p[word[start:end]]
        bwd = [0.0] * L + [1.0]              # bwd[i]: probability of word[i:]
        for start in range(L - 1, -1, -1):
            for end in range(start + 1, L + 1):
                if word[start:end] in p:
                    bwd[start] += p[word[start:end]] * bwd[end]
        if fwd[L] == 0:
            continue
        for start in range(L):
            for end in range(start + 1, L + 1):
                piece = word[start:end]
                if piece in p:
                    counts[piece] += n * fwd[start] * p[piece] * bwd[end] / fwd[L]
    return dict(counts)


def em_step(corpus, p):
    counts = expected_counts(corpus, p)
    total = sum(counts.values())
    return {t: c / total for t, c in counts.items() if c > 0}

**Why it works.** `fwd[i]` is Part 1's table, the probability of the first `i` letters, and `bwd[i]` is the
same sum for the letters from `i` to the end. The segmentations that use `piece` at
positions `start` to `end` have total probability `fwd[start] * p[piece] * bwd[end]`, and
dividing by the word's probability `fwd[L]` turns that into the probability, given the word,
that the piece is used there. Summing over positions and weighting by the word's count gives
the expected count, in $O(L^2)$ per word. Listing every segmentation gives the same numbers
more slowly. Counting only the Viterbi split would give every use to the best split, 7 uses
of ab in the example where the model expects 4, and tokens that are never best would die
out after one step. The check prints the gain in log-probability from one step: from the
seed's substring counts to probabilities that fit how the words actually split.

### Part 3: Pruning

In [ ]:
def removal_loss(corpus, p, token):
    without = {t: q for t, q in p.items() if t != token}
    loss = 0.0
    for word, n in corpus.items():
        if token not in word:
            continue                         # this word's probability does not change
        after = word_prob(word, without)
        if after == 0:
            return math.inf
        loss += n * (math.log(word_prob(word, p)) - math.log(after))
    return loss


def prune(corpus, p, n_remove):
    candidates = [t for t in p if len(t) > 1]
    loss = {t: removal_loss(corpus, p, t) for t in candidates}
    dropped = set(sorted(candidates, key=loss.get)[:n_remove])
    kept = {t: q for t, q in p.items() if t not in dropped}
    total = sum(kept.values())
    return {t: q / total for t, q in kept.items()}

**Why it works.** Removing a token only changes the probability of words that contain it, so the loop can
skip the others. Because the other tokens keep their probabilities, the loss measures what
the corpus loses when its segmentations through that token disappear, which is the rule the
lesson describes: keep the tokens that matter most to the likelihood. It is not the same as
keeping the most probable tokens. In the check, abc has the smallest probability of the
multi-letter tokens, but it is the best split of the corpus's commonest word, and removing it
costs $10 \ln(0.116 / 0.066) = 5.64$. bc is three times as probable, yet the word abc keeps
three other splits without it, so removing bc costs only $10 \ln(0.116 / 0.086) = 2.99$. Single letters
are never candidates, which is why every new word in the last lines can be split. The
trained vocabulary keeps whole frequent words such as player and played next to pieces such
as re, un, writ and ing, and the unseen words split into them, although no rule about
English prefixes or suffixes was given.